# Yamanalyst: self-supervised ball detector (GPU)

Fine-tunes a small YOLO on the tactical video, using the ATD ball (projected through the
homography) as automatic labels, then detects the ball on every graded frame where the tracking
lost it. Code: `mtah/video_ft.py`.

**Data rules — read before running**
- The match data lives only in your Google Drive (`MyDrive/MTAH/`) and this runtime.
- **Keep Colab's Gemini / AI features off for this notebook.** Do not ask an AI assistant about
  any cell or output.
- Cells print aggregate numbers only. Never add a cell that displays frames or crops.

Runtime → Change runtime type → **GPU** (T4 works; L4/A100 is faster).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/MTAH'
import os
for f in ('BAR-ATH_tactical.mp4', 'BAR-ATH_tactical_atd.xml', 'BAR-ATH_tactical_atd.txt',
          'BAR-ATH_tactical_homography.csv', 'BAR-ATH_tactical_smart_tagging.xml'):
    assert os.path.isfile(f'{DRIVE}/data/tactical/{f}'), f'missing {f} in MyDrive/MTAH/data/tactical'
assert os.path.isfile(f'{DRIVE}/mtah_code.zip'), 'missing MyDrive/MTAH/mtah_code.zip'
print('Drive OK')

In [ ]:
# local SSD copy (sequential video decode from Drive is slow), code, and the one dependency
!rm -rf /content/code && mkdir -p /content/code /content/data /content/work
!unzip -q -o "{DRIVE}/mtah_code.zip" -d /content/code
!rsync -a "{DRIVE}/data/tactical" /content/data/
!pip -q install ultralytics==8.3.253
os.environ['MTAH_DATA_DIR'] = '/content/data'
os.environ['MTAH_WORK'] = '/content/work'
os.environ['YOLO_OFFLINE'] = '1'
from ultralytics import settings
settings.update({'sync': False})     # no analytics leave the runtime
%cd /content/code
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!python -m mtah.data                # parse ATD + homography into cache/ (~10 s)

In [ ]:
!python -m mtah.video_ft dataset     # ~7.5k labelled crops from ATD-visible frames

In [ ]:
!python -m mtah.video_ft train yolo11s 30 1024 16 0     # base, epochs, imgsz, batch, device

In [ ]:
!python -m mtah.video_ft evaluate 0  # held-out recall / precision vs the ATD ball

In [ ]:
!python -m mtah.video_ft detect 0    # every graded missing-ball frame (+ public window)

In [ ]:
!cp /content/work/ball_det_ft.npz "{DRIVE}/ball_det_ft.npz"
!cp /content/work/ball_ft/weights/best.pt "{DRIVE}/ball_ft_best.pt"
print('saved to Drive: ball_det_ft.npz, ball_ft_best.pt')